In [ ]:
# Install required packages
!pip install -q \
    pyngrok       # For creating public URLs for local servers \
    transformers  # For loading and running the model \
    fastapi       # For creating the REST API \
    uvicorn       # ASGI server for FastAPI \
    nest_asyncio  # For running async code in Jupyter

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model Configuration
model_path = 'WaltonFuture/Diabetica-7B'  # Update with your model path

def load_model(model_path):
    """
    Load the Diabetica language model and tokenizer.
    
    Args:
        model_path (str): Path to the pre-trained model
    Returns:
        tuple: (model, tokenizer, device) - Loaded model, tokenizer and device
    """
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype="auto",
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    device = next(model.parameters()).device
    return model, tokenizer, device

def generate_response(user_prompt, max_new_tokens=100):
    """
    Generate model response for given user prompt.
    
    Args:
        user_prompt (str): Input text from user
        max_new_tokens (int): Maximum length of generated response
        
    Returns:
        str: Generated response from the model
    """
    system_message = "You are a helpful assistant."
    prompt = f"System: {system_message}\nUser: {user_prompt}\nAssistant:"
    
    # Tokenize and generate response
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=max_new_tokens,
        temperature=0.7,  # Controls randomness
        top_p=0.9,       # Nucleus sampling parameter
        do_sample=True   # Enable sampling
    )
    
    # Extract response from generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = (generated_text.split("Assistant:")[-1].strip() 
               if "Assistant:" in generated_text 
               else generated_text.strip())
    
    return response

# Initialize model, tokenizer and device
model, tokenizer, device = load_model(model_path)

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import os

# Define request and response models
class PromptRequest(BaseModel):
    prompt: str

class PromptResponse(BaseModel):
    response: str

# API Configuration
DEFAULT_MAX_LENGTH = 2000  # Maximum token length for responses
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"  # Replace with your token
API_PORT = 8000

# Create FastAPI instance
app = FastAPI()

@app.post("/generate", response_model=PromptResponse)
async def generate_text(request: PromptRequest):
    """
    Generate response for the given prompt using Diabetica model.
    
    Args:
        request (PromptRequest): Contains the user's prompt
    Returns:
        PromptResponse: Contains the model's response
    """
    try:
        response_text = generate_response(
            request.prompt, 
            max_new_tokens=DEFAULT_MAX_LENGTH
        )
        return {"response": response_text}
    except Exception as e:
        raise HTTPException(
            status_code=500, 
            detail=f"Error generating text: {str(e)}"
        )

def start_server():
    """Initialize and start the API server with ngrok tunnel."""
    # Allow nested asyncio loops (required for Jupyter)
    nest_asyncio.apply()
    
    # Configure ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(API_PORT)
    print(f"API is live at: {public_url}")
    
    # Start FastAPI server
    uvicorn.run(
        app, 
        host="0.0.0.0", 
        port=API_PORT
    )

# Start the server
if __name__ == "__main__":
    start_server()